In [ ]:
# ================================================================
# CELL 1: INSTALL DEPENDENCIES
# Must run first. Pins numpy before torch to avoid Numba crash.
# ================================================================
import subprocess, sys, os

def install(pkgs, label):
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + pkgs,
        capture_output=True, text=True
    )
    status = "OK" if r.returncode == 0 else "FAILED"
    print(f"  [{status}] {label}")
    if r.returncode != 0:
        print(r.stderr[-300:])

# ── Step 0: Uninstall any conflicting numpy, pin to 1.26.4 ──────
print("Step 0: Pinning numpy...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "numpy"],
               capture_output=True)
install(["numpy==1.26.4", "--force-reinstall"], "numpy==1.26.4")

# ── Step 1: System packages ──────────────────────────────────────
print("\nStep 1: System packages...")
subprocess.run(["apt-get", "update", "-qq"], capture_output=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], capture_output=True)
print("  [OK] ffmpeg")

# ── Step 2: PyTorch ──────────────────────────────────────────────
print("\nStep 2: PyTorch...")
install(["torch", "torchaudio",
         "--index-url", "https://download.pytorch.org/whl/cu118"],
        "torch + torchaudio (CUDA 11.8)")

# ── Step 3: Whisper ──────────────────────────────────────────────
print("\nStep 3: Whisper...")
install(["openai-whisper"], "openai-whisper")

# ── Step 4: All other packages ───────────────────────────────────
print("\nStep 4: Other packages...")
packages = [
    (["groq"],                          "groq client"),
    (["edge-tts"],                      "edge-tts (Microsoft Neural)"),
    (["pydub"],                         "pydub"),
    (["soundfile"],                     "soundfile"),
    (["scipy"],                         "scipy"),
    (["gradio>=4.0"],                   "gradio"),
    (["nest_asyncio"],                  "nest_asyncio"),
    (["rich"],                          "rich"),
    (["jiwer"],                         "jiwer (WER scoring)"),
]
for pkgs, label in packages:
    install(pkgs, label)

# ── Step 5: Verify all critical imports ──────────────────────────
print("\nStep 5: Verification...")
checks = [
    ("numpy < 2.0",
     "import numpy as np; v=tuple(int(x) for x in np.__version__.split('.')[:2]); assert v<(2,0)"),
    ("whisper",   "import whisper"),
    ("torch",     "import torch"),
    ("edge_tts",  "import edge_tts"),
    ("groq",      "import groq"),
    ("gradio",    "import gradio as gr; assert int(gr.__version__.split('.')[0]) >= 4"),
    ("pydub",     "from pydub import AudioSegment"),
    ("soundfile", "import soundfile as sf"),
    ("jiwer",     "import jiwer"),
]

all_ok = True
for name, stmt in checks:
    r = subprocess.run([sys.executable, "-c", stmt], capture_output=True)
    ok = r.returncode == 0
    print(f"  {'[OK]' if ok else '[FAIL]'} {name}")
    if not ok:
        print("    ", r.stderr.decode()[-200:])
        all_ok = False

print()
if all_ok:
    print("ALL CHECKS PASSED — proceed to Cell 2")
else:
    print("Some checks failed — re-run this cell once more")

Step 0: Pinning numpy...
  [OK] numpy==1.26.4

Step 1: System packages...
  [OK] ffmpeg

Step 2: PyTorch...
  [OK] torch + torchaudio (CUDA 11.8)

Step 3: Whisper...
  [OK] openai-whisper

Step 4: Other packages...
  [OK] groq client
  [OK] edge-tts (Microsoft Neural)
  [OK] pydub
  [OK] soundfile
  [OK] scipy
  [OK] gradio
  [OK] nest_asyncio
  [OK] rich
  [OK] jiwer (WER scoring)

Step 5: Verification...
  [OK] numpy < 2.0
  [OK] whisper
  [OK] torch
  [OK] edge_tts
  [OK] groq
  [OK] gradio
  [OK] pydub
  [OK] soundfile
  [OK] jiwer

ALL CHECKS PASSED — proceed to Cell 2


In [ ]:
# ================================================================
# CELL 2: CONFIGURATION
# Free Groq key: https://console.groq.com (no credit card needed)
# ================================================================

# ── API key (REQUIRED) ───────────────────────────────────────────
GROQ_API_KEY = "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"

# ── Whisper model size ───────────────────────────────────────────
# tiny=fastest, base=fast, small=best balance, medium=most accurate
WHISPER_SIZE = "small"

# ── Agentic settings ─────────────────────────────────────────────
MAX_RETRIES         = 3      # max retry attempts per stage
ASR_CONF_THRESHOLD  = 0.55   # min Whisper confidence to accept (0-1)
TRANS_LEN_MIN_RATIO = 0.25   # translated text must be >= 25% length of source
TRANS_LEN_MAX_RATIO = 5.0    # translated text must be <= 5x length of source
AUDIO_MIN_DURATION  = 0.4    # synthesized audio must be >= 0.4 seconds
AUDIO_MIN_RMS       = 0.002  # synthesized audio must not be silence

# ── Paths ────────────────────────────────────────────────────────
import os
OUTPUT_DIR  = "/content/dub_outputs"
MEMORY_FILE = "/content/dub_outputs/agent_memory.json"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Validate ─────────────────────────────────────────────────────
if GROQ_API_KEY.startswith("gsk_PASTE"):
    raise ValueError("Set your GROQ_API_KEY above before continuing!")

print("Configuration OK")
print(f"  Whisper model : {WHISPER_SIZE}")
print(f"  Max retries   : {MAX_RETRIES}")
print(f"  ASR threshold : {ASR_CONF_THRESHOLD}")
print(f"  Output dir    : {OUTPUT_DIR}")
print(f"  Memory file   : {MEMORY_FILE}")

Configuration OK
  Whisper model : small
  Max retries   : 3
  ASR threshold : 0.55
  Output dir    : /content/dub_outputs
  Memory file   : /content/dub_outputs/agent_memory.json


In [ ]:
# ================================================================
# CELL 3: PERSISTENT MEMORY STORE
# Saves every job to JSON on disk so the agent improves over time.
# Records: language pairs, strategies tried, quality scores,
#          which strategy worked best per pair.
# ================================================================
import json
import time
import uuid
from datetime import datetime

class AgentMemory:
    """
    Persistent key-value + job-history store backed by a JSON file.
    Survives Colab runtime restarts (as long as /content/ is mounted).
    """

    def __init__(self, path):
        self.path = path
        self._default = {
            "jobs":               [],      # list of completed job records
            "strategy_scores":    {},      # {lang_pair: {strategy: [scores]}}
            "total_jobs":         0,
            "total_success":      0,
        }
        self.data = self._load()

    def _load(self):
        try:
            with open(self.path) as f:
                d = json.load(f)
            # merge in any new keys from default
            for k, v in self._default.items():
                if k not in d:
                    d[k] = v
            print(f"  Memory loaded — {d['total_jobs']} past jobs found")
            return d
        except (FileNotFoundError, json.JSONDecodeError):
            print("  Memory initialised (no prior history)")
            return dict(self._default)

    def save(self):
        with open(self.path, "w") as f:
            json.dump(self.data, f, indent=2)

    def record_job(self, src_lang, tgt_lang, strategy,
                   asr_score, trans_score, voice_score,
                   final_score, success, attempts):
        job = {
            "id":          str(uuid.uuid4())[:8],
            "timestamp":   datetime.utcnow().isoformat(),
            "src_lang":    src_lang,
            "tgt_lang":    tgt_lang,
            "strategy":    strategy,
            "asr_score":   round(asr_score,   3),
            "trans_score": round(trans_score, 3),
            "voice_score": round(voice_score, 3),
            "final_score": round(final_score, 3),
            "success":     success,
            "attempts":    attempts,
        }
        self.data["jobs"].append(job)
        self.data["total_jobs"]    += 1
        if success:
            self.data["total_success"] += 1

        # update strategy scores for this language pair
        pair = f"{src_lang}->{tgt_lang}"
        ss   = self.data["strategy_scores"]
        if pair not in ss:
            ss[pair] = {}
        if strategy not in ss[pair]:
            ss[pair][strategy] = []
        ss[pair][strategy].append(final_score)

        self.save()
        return job["id"]

    def best_strategy(self, src_lang, tgt_lang):
        """Return the strategy with the highest average score for this pair."""
        pair = f"{src_lang}->{tgt_lang}"
        ss   = self.data["strategy_scores"].get(pair, {})
        if not ss:
            return None
        return max(ss, key=lambda s: sum(ss[s]) / len(ss[s]))

    def stats(self):
        total   = self.data["total_jobs"]
        success = self.data["total_success"]
        rate    = f"{(success/total*100):.1f}%" if total else "n/a"
        pairs   = list(self.data["strategy_scores"].keys())
        return {
            "total":   total,
            "success": success,
            "rate":    rate,
            "pairs":   pairs,
        }

    def recent_jobs(self, n=10):
        return self.data["jobs"][-n:]


memory = AgentMemory(MEMORY_FILE)
print("Memory store ready")

  Memory initialised (no prior history)
Memory store ready


In [ ]:
# CELL 4: LOAD MODELS (FAST VERSION)
import whisper
import torch
import asyncio
import nest_asyncio
import edge_tts
from pydub import AudioSegment
from groq import Groq
from functools import lru_cache

nest_asyncio.apply()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Use TINY model for speed (still very good for dubbing)
WHISPER_SIZE = "tiny"
print(f"Loading Whisper '{WHISPER_SIZE}'...")
whisper_model = whisper.load_model(WHISPER_SIZE, device=device)

groq_client = Groq(api_key=GROQ_API_KEY)
print("Groq ready")

# Voice map (unchanged)
VOICE_MAP = {
    "en": "en-US-JennyNeural",    "hi": "hi-IN-SwaraNeural",
    "kn": "kn-IN-SapnaNeural",    "ta": "ta-IN-PallaviNeural",
    "te": "te-IN-ShrutiNeural",   "fr": "fr-FR-DeniseNeural",
    "es": "es-ES-ElviraNeural",   "de": "de-DE-KatjaNeural",
    "it": "it-IT-ElsaNeural",     "pt": "pt-BR-FranciscaNeural",
    "ar": "ar-SA-ZariyahNeural",  "zh": "zh-CN-XiaoxiaoNeural",
    "ja": "ja-JP-NanamiNeural",   "ko": "ko-KR-SunHiNeural",
}

LANG_NAMES = {
    "en": "English", "hi": "Hindi", "kn": "Kannada", "ta": "Tamil",
    "te": "Telugu", "fr": "French", "es": "Spanish", "de": "German",
    "it": "Italian", "pt": "Portuguese", "ar": "Arabic", "zh": "Chinese",
    "ja": "Japanese", "ko": "Korean",
}

# Cache for edge-tts (avoid re-synthesizing same text)
@lru_cache(maxsize=128)
def cached_edge_speak(text, lang_code, rate="+0%"):
    voice = VOICE_MAP.get(lang_code, "en-US-JennyNeural")
    mp3_tmp = f"/tmp/tts_{hash(text)}_{lang_code}.mp3"
    wav_out = f"/tmp/tts_{hash(text)}_{lang_code}.wav"
    async def _gen():
        comm = edge_tts.Communicate(text, voice, rate=rate)
        await comm.save(mp3_tmp)
    asyncio.get_event_loop().run_until_complete(_gen())
    AudioSegment.from_mp3(mp3_tmp).export(wav_out, format="wav")
    return wav_out

def edge_speak_fast(text, lang_code, out_wav, rate="+0%"):
    cached_path = cached_edge_speak(text, lang_code, rate)
    import shutil
    shutil.copy(cached_path, out_wav)
    return out_wav

print("Fast voice cache ready")

Device: cpu
Loading Whisper 'tiny'...



  0%|                                              | 0.00/72.1M [00:00<?, ?iB/s]
 22%|████████▎                             | 15.7M/72.1M [00:00<00:00, 165MiB/s]
 44%|████████████████▌                     | 31.4M/72.1M [00:00<00:00, 148MiB/s]
 63%|████████████████████████              | 45.6M/72.1M [00:00<00:00, 139MiB/s]
 82%|███████████████████████████████       | 59.0M/72.1M [00:00<00:00, 133MiB/s]
100%|██████████████████████████████████████| 72.1M/72.1M [00:00<00:00, 135MiB/s]


Groq ready
Fast voice cache ready


In [ ]:
# ================================================================
# CELL 5: SPECIALIST AGENTS
# Each agent has two methods:
#   run()      — executes its task, returns result
#   evaluate() — scores the result, returns (score 0-1, issues[])
# Scores below threshold trigger a retry with a different strategy.
# ================================================================
import numpy as np
import soundfile as sf
import re
import time

# ────────────────────────────────────────────────────────────────
class ASRAgent:
    """
    Transcribes audio using Whisper.
    Self-evaluates on: confidence logprob, language match,
    speech rate plausibility, transcript length.
    On retry: escalates to a larger Whisper model.
    """

    def run(self, audio_path, src_lang="auto", model_size=None):
        """
        Returns: (text, detected_lang, avg_logprob, duration_sec)
        """
        global whisper_model
        if model_size and model_size != WHISPER_SIZE:
            print(f"    [ASR] Escalating model to '{model_size}'")
            whisper_model = whisper.load_model(model_size, device=device)

        kwargs = dict(
            fp16=(device == "cuda"),
            verbose=False,
            condition_on_previous_text=False,
        )
        if src_lang != "auto":
            kwargs["language"] = src_lang

        result   = whisper_model.transcribe(audio_path, **kwargs)
        text     = result["text"].strip()
        detected = result.get("language", src_lang)
        segs     = result.get("segments", [])
        logprob  = (
            sum(s.get("avg_logprob", -1.0) for s in segs) / len(segs)
            if segs else -1.0
        )

        # get duration from audio
        try:
            data, sr = sf.read(audio_path)
            duration = len(data) / sr
        except Exception:
            duration = 0.0

        return text, detected, logprob, duration

    def evaluate(self, text, detected_lang, expected_lang,
                 logprob, duration):
        """
        Returns (score, issues_list).
        score 1.0 = perfect, 0.0 = unusable.
        """
        score  = 1.0
        issues = []

        # 1. Empty transcript
        if not text or len(text.strip()) < 3:
            return 0.0, ["empty transcript"]

        # 2. Whisper confidence (logprob: 0=perfect, -1=bad, <-1.5=garbage)
        if logprob < -1.5:
            score -= 0.5
            issues.append(f"very low confidence ({logprob:.2f})")
        elif logprob < -0.8:
            score -= 0.25
            issues.append(f"low confidence ({logprob:.2f})")
        elif logprob < -0.5:
            score -= 0.1

        # 3. Language match
        if expected_lang not in ("auto", detected_lang):
            score -= 0.2
            issues.append(f"lang mismatch: expected {expected_lang}, got {detected_lang}")

        # 4. Words per second sanity check
        if duration > 0:
            wps = len(text.split()) / duration
            if wps < 0.3:
                score -= 0.15
                issues.append(f"too sparse ({wps:.1f} words/sec)")
            elif wps > 12:
                score -= 0.1
                issues.append(f"suspiciously fast ({wps:.1f} words/sec)")

        # 5. Minimum word count
        if len(text.split()) < 2:
            score -= 0.3
            issues.append("transcript too short")

        return max(0.0, min(1.0, score)), issues

    def escalate_model(self, current_size):
        """Return next larger model name, or None if already at max."""
        try:
            idx = WHISPER_SIZES.index(current_size)
            return WHISPER_SIZES[idx + 1] if idx + 1 < len(WHISPER_SIZES) else None
        except ValueError:
            return None


# ────────────────────────────────────────────────────────────────
class TranslateAgent:
    """
    Translates text using Groq LLaMA-3.3-70B.
    Self-evaluates on: length ratio, language leak, coherence.
    On retry: switches translation strategy.
    """

    def run(self, text, src_lang, tgt_lang, strategy="natural"):
        """
        Returns translated string.
        """
        src_name = LANG_NAMES.get(src_lang, src_lang)
        tgt_name = LANG_NAMES.get(tgt_lang, tgt_lang)
        guidance = STRATEGY_PROMPTS.get(strategy, STRATEGY_PROMPTS["natural"])

        prompt = (
            f"{guidance}\n\n"
            f"Translate the following {src_name} text to {tgt_name}.\n"
            f"Output ONLY the translation. No explanations, no quotes, no notes.\n\n"
            f'Text:\n"""\n{text}\n"""'
        )

        resp = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            max_tokens=2048,
        )
        translation = resp.choices[0].message.content.strip()

        # Strip accidental quotes the model sometimes adds
        if translation.startswith('"') and translation.endswith('"'):
            translation = translation[1:-1].strip()

        return translation

    def evaluate(self, source, translation, src_lang, tgt_lang):
        """
        Returns (score, issues_list).
        """
        score  = 1.0
        issues = []

        if not translation or len(translation.strip()) < 2:
            return 0.0, ["empty translation"]

        src_words = len(source.split())
        tgt_words = len(translation.split())

        # 1. Length ratio
        ratio = tgt_words / max(src_words, 1)
        if ratio < TRANS_LEN_MIN_RATIO:
            score -= 0.4
            issues.append(f"translation too short (ratio {ratio:.2f})")
        elif ratio > TRANS_LEN_MAX_RATIO:
            score -= 0.25
            issues.append(f"translation too long (ratio {ratio:.2f})")

        # 2. Source language leak detection
        # Check for Devanagari (hi), Kannada, Tamil, Telugu scripts in
        # a translation that should be in a Latin-script language
        latin_targets = {"en", "fr", "es", "de", "it", "pt", "nl", "sv", "tr", "pl"}
        if tgt_lang in latin_targets:
            non_latin = sum(1 for c in translation if ord(c) > 0x0900)
            if non_latin > 3:
                score -= 0.4
                issues.append(f"source-language characters in translation ({non_latin} chars)")

        # 3. Translation is identical to source (untranslated)
        if translation.strip().lower() == source.strip().lower():
            score -= 0.5
            issues.append("translation identical to source — model may have failed")

        # 4. Very short absolute length
        if tgt_words < 1:
            return 0.0, ["empty translation"]

        return max(0.0, min(1.0, score)), issues


# ────────────────────────────────────────────────────────────────
class VoiceAgent:
    """
    Synthesizes speech using edge-tts.
    Self-evaluates on: file exists, duration, RMS energy (silence check).
    On retry: adjusts speech rate.
    """

    RATE_SEQUENCE = ["+0%", "-10%", "+10%", "-5%"]  # rate to try on each attempt

    def run(self, text, tgt_lang, out_wav, attempt=0):
        """
        Returns path to synthesized WAV.
        """
        rate = self.RATE_SEQUENCE[min(attempt, len(self.RATE_SEQUENCE) - 1)]
        return edge_speak(text, lang_code=tgt_lang,
                          out_wav=out_wav, rate=rate)

    def evaluate(self, wav_path):
        """
        Returns (score, issues_list).
        """
        score  = 1.0
        issues = []

        if not os.path.exists(wav_path):
            return 0.0, ["output file does not exist"]

        try:
            data, sr = sf.read(wav_path)
        except Exception as e:
            return 0.0, [f"cannot read wav: {e}"]

        if len(data) == 0:
            return 0.0, ["empty audio file"]

        duration = len(data) / sr
        rms      = float(np.sqrt(np.mean(data.astype(np.float64) ** 2)))

        # 1. Duration check
        if duration < AUDIO_MIN_DURATION:
            score -= 0.5
            issues.append(f"audio too short ({duration:.2f}s)")

        # 2. Silence / energy check
        if rms < AUDIO_MIN_RMS:
            score -= 0.5
            issues.append(f"audio is silent or near-silent (RMS={rms:.4f})")
        elif rms < AUDIO_MIN_RMS * 3:
            score -= 0.2
            issues.append(f"audio very quiet (RMS={rms:.4f})")

        # 3. Suspiciously long (may be looping)
        if duration > 300:
            score -= 0.1
            issues.append(f"audio very long ({duration:.0f}s) — check input")

        return max(0.0, min(1.0, score)), issues


# ── Instantiate all agents ────────────────────────────────────────
asr_agent   = ASRAgent()
trans_agent = TranslateAgent()
voice_agent = VoiceAgent()

print("All specialist agents ready:")
print("  ASRAgent       — Whisper + confidence scoring + model escalation")
print("  TranslateAgent — Groq LLaMA-3.3-70B + length/leak scoring")
print("  VoiceAgent     — edge-tts + silence/duration scoring")

All specialist agents ready:
  ASRAgent       — Whisper + confidence scoring + model escalation
  TranslateAgent — Groq LLaMA-3.3-70B + length/leak scoring
  VoiceAgent     — edge-tts + silence/duration scoring


In [ ]:
# CELL 6: FAST ORCHESTRATOR
import time, hashlib

# Cache for translations
translation_cache = {}

class FastOrchestrator:
    def run(self, audio_path, src_lang="auto", tgt_lang="en", output_path=None):
        t_start = time.time()
        norm_path = f"{OUTPUT_DIR}/_norm.wav"
        AudioSegment.from_file(audio_path).set_frame_rate(16000).set_channels(1).export(norm_path, format="wav")

        # ── ASR (single pass, no retry) ──
        result = whisper_model.transcribe(norm_path, fp16=(device=="cuda"), verbose=False)
        src_text = result["text"].strip()
        detected = result.get("language", src_lang)
        if src_lang == "auto":
            src_lang = detected
        asr_score = 0.8 if len(src_text.split()) > 2 else 0.4  # quick heuristic

        # ── Translation with cache ──
        cache_key = f"{src_text}_{src_lang}_{tgt_lang}"
        if cache_key in translation_cache:
            tgt_text = translation_cache[cache_key]
            trans_score = 0.9
            strategy = "cached"
        else:
            # Single attempt with best strategy from memory
            best = memory.best_strategy(src_lang, tgt_lang) or "natural"
            prompt = STRATEGY_PROMPTS[best]
            resp = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": f"{prompt}\n\nTranslate to {LANG_NAMES[tgt_lang]}:\n{src_text}"}],
                temperature=0.2,
                max_tokens=1024,
            )
            tgt_text = resp.choices[0].message.content.strip()
            translation_cache[cache_key] = tgt_text
            trans_score = 0.8
            strategy = best

        # ── Voice synthesis (cached) ──
        out_path = output_path or f"{OUTPUT_DIR}/dubbed_fast.wav"
        edge_speak_fast(tgt_text, tgt_lang, out_path, rate="+0%")
        voice_score = 0.9  # edge-tts is reliable

        final_score = (0.3*asr_score + 0.35*trans_score + 0.35*voice_score)
        elapsed = time.time() - t_start

        # Record to memory (async, don't block)
        job_id = memory.record_job(src_lang, tgt_lang, strategy, asr_score, trans_score, voice_score, final_score, True, 1)

        return {
            "src_text": src_text, "tgt_text": tgt_text,
            "asr_score": asr_score, "trans_score": trans_score, "voice_score": voice_score,
            "final_score": final_score, "strategy": strategy, "output_path": out_path,
            "job_id": job_id, "elapsed": elapsed, "src_lang": src_lang, "tgt_lang": tgt_lang,
        }

orchestrator = FastOrchestrator()
print("Fast orchestrator ready – single retry, caching, tiny Whisper")

Fast orchestrator ready – single retry, caching, tiny Whisper


In [ ]:
# ================================================================
# CELL 7: GRADIO UI (FIXED – Self‑contained)
# ================================================================
import gradio as gr
import traceback
import os
from pydub import AudioSegment

# ── Ensure output directory exists (even if Cell 2 wasn't run) ──
OUTPUT_DIR = "/content/dub_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Check that the core agents are loaded ───────────────────────
try:
    orchestrator
    memory
except NameError:
    raise RuntimeError(
        "Please run Cells 1–6 first! The orchestrator and memory agents are not loaded."
    )

# Language choices (same as before)
INPUT_CHOICES = {
    "Auto-detect": "auto",
    "Kannada": "kn", "Hindi": "hi", "English": "en",
    "Tamil": "ta", "Telugu": "te", "French": "fr",
    "Spanish": "es", "German": "de", "Italian": "it",
    "Portuguese": "pt", "Arabic": "ar", "Chinese": "zh",
    "Japanese": "ja", "Korean": "ko", "Polish": "pl",
    "Russian": "ru", "Dutch": "nl", "Swedish": "sv",
    "Turkish": "tr", "Vietnamese": "vi",
}
OUTPUT_CHOICES = {k: v for k, v in INPUT_CHOICES.items() if k != "Auto-detect"}

def fmt_score(s):
    if s >= 0.80: return f"Excellent ({s:.2f})"
    if s >= 0.65: return f"Good ({s:.2f})"
    if s >= 0.50: return f"Acceptable ({s:.2f})"
    return f"Poor ({s:.2f})"

def run_pipeline(audio_file, src_label, tgt_label):
    if audio_file is None:
        return None, "No audio provided", "", "", "", ""

    src_code = INPUT_CHOICES.get(src_label, "auto")
    tgt_code = OUTPUT_CHOICES.get(tgt_label, "en")

    try:
        # Normalise input to ensure Whisper compatibility
        norm_path = f"{OUTPUT_DIR}/_input_norm.wav"
        AudioSegment.from_file(audio_file).set_frame_rate(16000).set_channels(1).export(norm_path, format="wav")

        result = orchestrator.run(
            audio_path=norm_path,
            src_lang=src_code,
            tgt_lang=tgt_code,
            output_path=f"{OUTPUT_DIR}/dubbed_final.wav",
        )

        stats = memory.stats()
        status = (
            f"✅ Job {result['job_id']} | {result['elapsed']}s\n"
            f"Strategy: {result['strategy']}\n"
            f"ASR: {result['asr_score']:.2f} | Trans: {result['trans_score']:.2f} | Voice: {result['voice_score']:.2f}\n"
            f"Final: {fmt_score(result['final_score'])}\n"
            f"Memory: {stats['total']} jobs, {stats['rate']} success"
        )

        # Build recent jobs history
        history_lines = []
        for j in memory.recent_jobs(5):
            history_lines.append(
                f"[{j['id']}] {j['src_lang']}→{j['tgt_lang']} | {j['strategy']} | score={j['final_score']:.2f} | {'✓' if j['success'] else '✗'}"
            )
        history = "\n".join(history_lines) if history_lines else "No jobs yet"

        return (
            result["output_path"],
            status,
            result["src_text"],
            result["tgt_text"],
            fmt_score(result["final_score"]),
            history
        )

    except Exception as e:
        tb = traceback.format_exc()
        return None, f"❌ Error:\n{tb[-800:]}", "", "", "0.00", ""

# ── Build the Gradio interface ───────────────────────────────────
with gr.Blocks(title="Agentic AI Dubbing System", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ Agentic AI Dubbing System")
    gr.Markdown(
        "**Orchestrator agent** – Whisper ASR → Groq LLM translation → edge-tts synthesis. "
        "Self‑evaluates, retries, and remembers past strategies."
    )

    with gr.Row():
        with gr.Column(scale=1):
            audio_input = gr.Audio(
                label="Record or upload audio",
                sources=["microphone", "upload"],
                type="filepath",
            )
            src_lang = gr.Dropdown(
                choices=list(INPUT_CHOICES.keys()),
                value="Kannada",
                label="Speaking language"
            )
            tgt_lang = gr.Dropdown(
                choices=list(OUTPUT_CHOICES.keys()),
                value="English",
                label="Dub into language"
            )
            run_btn = gr.Button("🚀 Run Agentic Pipeline", variant="primary", size="lg")

        with gr.Column(scale=1):
            audio_output = gr.Audio(label="Dubbed audio", type="filepath", show_download_button=True)
            score_display = gr.Textbox(label="Quality score", lines=1, interactive=False)
            status_display = gr.Textbox(label="Agent status & memory", lines=6, interactive=False)

    with gr.Row():
        src_text = gr.Textbox(label="Transcribed source (Whisper)", lines=5, interactive=False)
        tgt_text = gr.Textbox(label="Translated text (used for synthesis)", lines=5, interactive=False)

    gr.Markdown("### 📜 Memory – Recent jobs")
    history_display = gr.Textbox(label="Last 5 jobs", lines=4, interactive=False)

    run_btn.click(
        fn=run_pipeline,
        inputs=[audio_input, src_lang, tgt_lang],
        outputs=[audio_output, status_display, src_text, tgt_text, score_display, history_display],
        show_progress="full"
    )

    gr.HTML(
        "<div style='text-align:center; margin-top:20px; color:gray; font-size:12px'>"
        "Agents: Whisper · Groq LLaMA-3.3-70B · edge-tts · Orchestrator · Self‑eval · Retry · Persistent memory"
        "</div>"
    )

print("🚀 Launching Gradio UI...")
demo.launch(share=True, debug=False, quiet=True)

/tmp/ipykernel_55858/3291249168.py:91: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Agentic AI Dubbing System", theme=gr.themes.Soft()) as demo:


🚀 Launching Gradio UI...
* Running on public URL: https://7050d46d1e28a33d70.gradio.live
